# Study 918 — Creation Halt 🚧

**When a fund stops printing new shares, does its price float free — and can you trade it?**

An exchange-traded product is glued to the value of what it holds by exactly one
mechanism: professional dealers create new shares when the price is rich and hand shares
back when it is cheap. Switch the creation side off and the glue only works one way —
supply is frozen, demand is not, and the price should rise to a premium until issuance
restarts. Freeze *redemptions* instead and the same logic runs in reverse, into a
discount.

We test that on **six hardcoded, publicly reported suspensions**: UNG (2009), USO (2020),
VXX and OIL (the same Barclays announcement, 2022), BITO (2021, a capacity constraint —
flagged as the weak one) and GBTC (2015–2024, a redemption freeze). Each capped fund is
measured against an **uncapped** instrument tracking the same thing.

*Real numbers below are the frozen headline (`docs/results.md`, fingerprint
`696cca301c40`, as-of 2026-06-30); the only live cells run the offline synthetic control and
say so.*


## 1. The idea, and the one honest measuring stick

To say a fund is 'trading rich' you need something to compare it against. The funds' own daily net-asset values are not published anywhere free, so we use the next best thing: an **uncapped** instrument that holds the same thing. For VXX that is VIXY — a different sponsor's fund tracking the *identical* VIX-futures index, which was never suspended. For GBTC it is bitcoin itself.

For the oil and gas funds there is no such twin. The nearest thing sits at a *different point on the futures curve*, so the comparison mixes the premium we want with the roll cost we don't. Remember that split — it turns out to be the whole story.

> 🔬 **For the quants:** the object is the signed daily log-return spread, `direction * (Δlog fund − Δlog twin)`, with `direction = −1` for the redemption freeze so all six events point the same way. It is self-financing, so it is an excess-of-cash quantity by construction.

## 2. The one case where you can see it perfectly

In March 2022 Barclays announced it had issued more VXX notes than it had registered and stopped selling new ones. VIXY, holding the same index, carried on as normal. So for five months the market ran a controlled experiment.

In [1]:
R = dict(vxx_ann_day=5.81, vxx_car20=15.41, vxx_pct_indep=0.99, vxx_nplacebo_indep=99,
         vxx_in_days=101, vxx_in_total=19.59, vxx_fade=-18.48, vxx_fade_pct_indep=0.0,
         vxx_path={5: 11.42, 10: 19.1, 20: 15.41, 40: -1.06, 60: 1.79, 101: 17.89})
print('VXX vs VIXY, 2022 suspension (frozen real-tape numbers)')
print('  move on the announcement day itself : %+.2f%% (we do NOT claim it - one day of lag)'
      % R['vxx_ann_day'])
print('  next 20 sessions, richer than VIXY  : %+.2f%%  (richer than %.1f%% of the %d independent 20-day windows)'
      % (R['vxx_car20'], R['vxx_pct_indep']*100, R['vxx_nplacebo_indep']))
print('  across the whole %d-session freeze   : %+.2f%%'
      % (R['vxx_in_days'], R['vxx_in_total']))
print('  20 sessions after issuance restarts : %+.2f%%  (cheaper than every other window)'
      % R['vxx_fade'])
print()
print('  but the premium ROUND-TRIPS while the halt is still on:')
for k in sorted(R['vxx_path']):
    print('    after %3d sessions: %+7.2f%%' % (k, R['vxx_path'][k]))

VXX vs VIXY, 2022 suspension (frozen real-tape numbers)
  move on the announcement day itself : +5.81% (we do NOT claim it - one day of lag)
  next 20 sessions, richer than VIXY  : +15.41%  (richer than 99.0% of the 99 independent 20-day windows)
  across the whole 101-session freeze   : +19.59%
  20 sessions after issuance restarts : -18.48%  (cheaper than every other window)

  but the premium ROUND-TRIPS while the halt is still on:
    after   5 sessions:  +11.42%
    after  10 sessions:  +19.10%
    after  20 sessions:  +15.41%
    after  40 sessions:   -1.06%
    after  60 sessions:   +1.79%
    after 101 sessions:  +17.89%


Out by 15%, back by 18%, on the two dates the theory names. If the story were going to be true anywhere, this is what it looks like.

Two cautions we have to put next to it straight away. **First**, the honest denominator is 99, not 1,978: a 'window' that slides forward one day at a time overlaps its neighbour by 19 days in 20, so eight and a half years of tape hold about 99 genuinely independent 20-day windows. **Second**, look at the last block of that output — the premium is not a steady build. It is up 19% by day 10, back to *minus* 1% by day 40, and up again by the end. Anyone holding it was on a rollercoaster, not an escalator.

## 3. And the case that would have taken your arm off

April 2020. USO announced it had run out of registered shares the day after oil settled at minus $37. Buy the halted fund, short an oil fund that was still creating shares — the same trade — and in **six sessions** you were down **-19.5%** net. The fund was not floating up to a premium; it was being forced to rebuild its entire portfolio in the middle of the worst week the oil market has ever had.

That is the problem in one line: the announcement tells you the arbitrage is broken. It does not tell you **which way**.

## 4. Six events, no pattern

Line all five dated announcements up and measure the same 20 sessions after each (starting the day *after*, so nobody is trading on information they could not have had):

In [2]:
car20 = {'UNG-2009': -3.3, 'USO-2020': -6.0, 'VXX-2022': 15.41, 'OIL-2022': -7.03, 'BITO-2021': -0.95}
z20 = {'UNG-2009': -0.24, 'USO-2020': -1.63, 'VXX-2022': 13.84, 'OIL-2022': -2.55, 'BITO-2021': -0.36}
for k in car20:
    verdict = 'as predicted' if z20[k] > 2 else ('opposite' if z20[k] < -2 else 'nothing')
    print('  %-10s %+7.2f%%   (%s)' % (k, car20[k], verdict))
print('\npooled across the five: mean standardised move %+.2f, t = %+.2f, positive in %d of 5'
      % (1.81, 0.6, 1))

  UNG-2009     -3.30%   (nothing)
  USO-2020     -6.00%   (nothing)
  VXX-2022    +15.41%   (as predicted)
  OIL-2022     -7.03%   (opposite)
  BITO-2021    -0.95%   (nothing)

pooled across the five: mean standardised move +1.81, t = +0.60, positive in 1 of 5


A *t* of **+0.60** is the statistical equivalent of a shrug. Take VXX out and the average flips to **-1.19** (*t* = -2.17) — i.e. the remaining halted funds got *cheaper*, not richer.

> 🔬 **For the quants:** each event's CAR is standardised against that pair's own distribution of every other 20-day window (1,000–4,900 controls), so a 3% move in the gas pair and a 3% move in the VIX pair are not treated as the same event.

## 5. The one thing that does travel: the collapse afterwards

The two funds that verifiably *did* carry a premium both handed it back violently once the presses restarted — UNG **-28.3%** and VXX **-18.5%** in 20 sessions, both in the extreme tail of their own history. That is the mechanism doing exactly what it should. It is still only 3 of 6 events pointing the right way (*t* = -0.96), because the other three never had a premium to give back in the first place.

## 6. Why you cannot bank it

The trade is: buy the capped fund, sell short the uncapped twin, hold until issuance resumes. Commissions are trivial on a five-month hold. **Borrow is not** — and the one instrument you must borrow is, by construction, the squeezed one.

- median result across the six events: **-4.29%** net, 2 of 6 profitable;
- the average looks positive (**+17.31%**) only because GBTC's 8.7-year discount counts as one 'trade'. Drop it and the average is **-4.51%**, 1 of 5 profitable;
- at a 30%/yr borrow rate — what a genuinely hard-to-borrow capped note costs — the average is **-27.26%**.

## 6b. And the part that quietly does all the work: knowing when to get out

Every number above holds the position **until issuance resumes** — a date nobody standing at the announcement could possibly have known. VXX's halt lasted 101 sessions; USO's lasted 6. So ask the fair question instead: what happens if you just buy on the announcement and sell 60 sessions later, like a real person with a calendar and no crystal ball?

In [3]:
hind = {'UNG-2009': -12.22, 'USO-2020': -19.52, 'VXX-2022': 17.8, 'OIL-2022': -3.78, 'GBTC-2024': 126.4, 'BITO-2021': -4.8}
blind = {'UNG-2009': -32.98, 'USO-2020': -6.45, 'VXX-2022': 0.57, 'OIL-2022': -2.14, 'GBTC-2024': 96.62, 'BITO-2021': -8.49}
bd = 60
print(f"{'event':<11s}{'told the end':>15s}{('blind ' + str(bd) + 'd'):>15s}")
for k in hind:
    print(f'{k:<11s}{hind[k]:>14.2f}%{blind[k]:>14.2f}%')
print()
print(f"{'median':<11s}{-4.29:>14.2f}%"
      f"{-4.29:>14.2f}%")

event         told the end      blind 60d
UNG-2009           -12.22%        -32.98%
USO-2020           -19.52%         -6.45%
VXX-2022            17.80%          0.57%
OIL-2022            -3.78%         -2.14%
GBTC-2024          126.40%         96.62%
BITO-2021           -4.80%         -8.49%

median              -4.29%         -4.29%


Look at the VXX row. Held to the day issuance actually restarted: **+17.80%**. Held for a fixed 60 sessions because that is all you could have decided in advance: **+0.57%**.

The entire profit of the single cleanest example in this study was the exit date — and the exit date was hindsight. That, more than the borrow, is why this is a Mirage.

## 7. Live check — is the measuring machine honest? (offline synthetic)

The cells below are **synthetic**, not the real tape. We build six imaginary fund/twin pairs, plant a premium that builds during a halt and fades after, and check the same code finds it — then switch the premium off and check the code finds nothing.

In [4]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from creation_halt import strategy as st
def avg(ss):
    r = [st.synthetic_detect(ss, seed=918 + 11*s) for s in range(8)]
    return (np.mean([x['mean_in_bps'] for x in r]),
            np.mean([x['mean_fade_z'] for x in r]))
pl_bps, pl_fade = avg(1.0)
nl_bps, nl_fade = avg(0.0)
print('planted halt premium : drift while halted %+.2f bps/day, fade afterwards z %+.2f (should fire)'
      % (pl_bps, pl_fade))
print('nothing planted      : drift while halted %+.2f bps/day, fade afterwards z %+.2f (should be ~0)'
      % (nl_bps, nl_fade))
print('(each line is the average of 8 independent synthetic worlds)')

planted halt premium : drift while halted +10.42 bps/day, fade afterwards z -4.73 (should fire)
nothing planted      : drift while halted -0.66 bps/day, fade afterwards z +0.02 (should be ~0)
(each line is the average of 8 independent synthetic worlds)


## Verdict

- **Signal — Mixed.** The mechanism is real and you can watch it happen: VXX out **+15.4%** and back **-18.5%** against an identical uncapped fund, at the extremes of its own history — percentile 0.990 and 0.000 of the 99 *independent* 20-day windows it has. We look at 30 such comparisons across the study, so neither of those two tails would impress on its own; what does is that both happened, in the two directions predicted, on the two dates named. But it does not generalise — pooled *t* = **+0.60**, 1/5 positive, and USO's halt went the other way entirely (-19.5% net on the same trade in six sessions). Our event list is also a *survivor's* list: it contains the halts that got reported, and the two most famous (TVIX 2012, the original VXX note) are missing because those instruments were delisted and their tapes are gone.
- **Tradability — Mirage.** Median net **-4.29%**, 2/6 profitable. And the flagship winner evaporates the moment you take away the crystal ball: VXX pays **+17.80%** if you are told when the halt ends and **+0.57%** if you are not. On top of that, the number that decides the trade — the borrow on a squeezed, capped fund — is one nobody publishes.